# Pokémon TCG Battle Agent — Phase 3 Colab Pro Training & RL Pipeline

Thin Colab notebook wrapping the repository CLI (`python -m src.cli ...`).
- Dataset: `kaggle/the-pokemon-company-ptcg-ai-battle-challenge-playground-episodes-2026-11-13` (Version 1, up to all `1,644` episodes)
- Simulator: CABT Engine (`cg` package)
- Target Runtime: **Colab Pro GPU (`T4 High-RAM` / `L4` / `A100`)**
- Click **Runtime -> Change runtime type -> T4 GPU** (High-RAM recommended for full 1,644 episodes), then **Runtime -> Run all**.

In [ ]:
# 1. Check GPU accelerator, RAM, disk space, and mount Google Drive for persistent checkpoints
!nvidia-smi
!df -h .
import os, torch
print('PyTorch:', torch.__version__, 'CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST_DIR = '/content/drive/MyDrive/ptcg_phase3_checkpoints'
except Exception:
    PERSIST_DIR = './checkpoints'
os.makedirs(PERSIST_DIR, exist_ok=True)
print('Persistent checkpoint dir:', PERSIST_DIR)

In [ ]:
# 2. Install dependencies
!pip install -q kagglehub kaggle pytest --no-warn-script-location

In [ ]:
# 3. Download full 1,644 episodes (or adjust --max-episodes for faster runs) and stream-preprocess into shards
MAX_EPISODES = 1644
!python -m src.cli download-data --out-dir data/replays --max-episodes {MAX_EPISODES}
!python -m src.cli preprocess --replay-dir data/replays --out-dir data/processed --seed 42 --shard-size 15000

In [ ]:
# 4. Train BC Policy with Set-Context Attention & Conditional Multi-Select on GPU (sharded streaming)
!python -m src.cli train --data-dir data/processed --checkpoint-dir {PERSIST_DIR} --epochs 15 --batch-size 128 --lr 0.002 --seed 42 --device cuda

# Verify checkpoint resume (+2 epochs) with monotonic CosineAnnealingLR
!python -m src.cli train --data-dir data/processed --checkpoint-dir {PERSIST_DIR} --epochs 17 --batch-size 128 --lr 0.002 --seed 42 --device cuda --resume {PERSIST_DIR}/last_model.pt

In [ ]:
# 5. RL / Self-Play Fine-Tuning (PPO + KL-to-BC) across all 3 decks in CABT
!python -m src.cli train-rl --bc-checkpoint {PERSIST_DIR}/best_bc_model.pt --checkpoint-dir {PERSIST_DIR} --updates 12 --episodes-per-update 18 --lr 0.0003 --kl-coef 0.05 --eval-games 18 --seed 42 --device cuda

# Sync checkpoints back to local ./checkpoints/
import shutil
if os.path.abspath(PERSIST_DIR) != os.path.abspath('./checkpoints'):
    os.makedirs('./checkpoints', exist_ok=True)
    for fn in ['best_model.pt', 'best_bc_model.pt', 'best_rl_model.pt', 'last_model.pt', 'last_rl_model.pt', 'train_summary.json', 'rl_summary.json']:
        src_f = os.path.join(PERSIST_DIR, fn)
        if os.path.exists(src_f):
            shutil.copy2(src_f, os.path.join('./checkpoints', fn))

In [ ]:
# 6. CPU Smoke Test, Mirror + Cross-Deck Matrix Evaluation, Latency Benchmark, and Pytest
!python -m src.cli smoke-test --deck deck.csv --checkpoint checkpoints/best_model.pt --seed 42
!python -m src.cli evaluate --checkpoint checkpoints/best_model.pt --data-dir data/processed --games 42 --cross-games 6 --seed 42 --out-file checkpoints/eval_results.json
!python -m pytest -p no:cacheprovider -v tests/test_pipeline.py